# 🟢 2.1 Read-First Reconnaissance: Auditing Legacy Codebases Without File Mutations
**Course:** Agentic CLI Engineering - Complete Claude Code Masterclass  
**Module:** Section 2 — Interactive Session Mastery & Context Hygiene  
**Shared Section 2 Project:** `./order_service` (Single Evolving Project Across Section 2)  
**OpenRouter Model:** `deepseek/deepseek-v4.1-flash-1017`  

---

## 🏛️ Architectural Objective
When dropping into an undocumented legacy repository, the worst mistake an engineer can make is prompting an AI to *"Fix the slow endpoint"* on Turn 1. Without mapping the repository's hidden coupling first, premature file mutations break downstream callers.

In **Read-First Reconnaissance**, we lock Claude Code strictly to its **Sensor Triad (`--allowedTools "Read,Grep,Glob"`)**—stripping away `Edit`, `Write`, and `Bash` so the agent maps call graphs and pinpoints root-cause defects with **zero file mutations on disk**.

In this hands-on lab, you will:
1. **Deploy our Shared Section 2 Microservice (`./order_service`)**, bind `.claude/settings.json` to OpenRouter (`deepseek/deepseek-v4.1-flash-1017`), and lock a clean Git baseline (`v2.1-baseline`).
2. **Execute `claude` Read-Only Reconnaissance from the Claude CLI Terminal** to generate an end-to-end ASCII Dependency Call Graph.
3. **Run a Forensic Security & Performance Audit from the Claude CLI Terminal** to uncover a critical JWT signature bypass (`auth/jwt_guard.py`) and a `2N+1` SQL query bottleneck (`services/report_builder.py`).
4. **Mathematically Verify Zero Disk Mutations** using `git status --porcelain` and `git diff`.

---
## Phase 1: Deploying the Shared Section 2 Microservice (`./order_service`) & Git Baseline
Let's scaffold `./order_service`—our persistent Section 2 multi-folder Python backend (`routes/`, `auth/`, `services/`, `db/`, `utils/`) that has zero documentation and is suffering from production timeouts and a suspected security hole. We will initialize a Git repository, configure `.claude/settings.json` with `deepseek/deepseek-v4.1-flash-1017`, and commit the baseline so we can audit file integrity after Claude inspects it.

In [1]:
%%bash
# 0. Auto-sync required packages from root requirements.txt
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")
REQ_FILE=$( [ -f "requirements.txt" ] && echo "requirements.txt" || echo "../requirements.txt" )
if [ -f "$REQ_FILE" ]; then
    echo -e "\033[0;36m[SETUP] Syncing Python dependencies from $REQ_FILE...\033[0m"
    $PY_BIN -m pip install -q -r "$REQ_FILE"
fi

rm -rf order_service
mkdir -p order_service/routes order_service/auth order_service/services order_service/db order_service/utils order_service/.claude
cd order_service

cat << 'EOF' > .gitignore
__pycache__/
*.pyc
.pytest_cache/
.env*
.claude/
EOF

# Bind OpenRouter Gateway + deepseek/deepseek-v4.1-flash-1017 from Lecture 1.2 settings
SAMPLE_SETTINGS="../../Foundation & Memory Architecture/lab_1_2_workspace/.claude_sample/settings.json"
if [ -f "$SAMPLE_SETTINGS" ]; then
    cp "$SAMPLE_SETTINGS" .claude/settings.json
else
    cat << 'EOF' > .claude/settings.json
{
  "env": {
    "ANTHROPIC_BASE_URL": "https://openrouter.ai/api",
    "ANTHROPIC_AUTH_TOKEN": "sk-or-v1-your-openrouter-key",
    "ANTHROPIC_API_KEY": "",
    "ANTHROPIC_MODEL": "deepseek/deepseek-v4.1-flash-1017",
    "ANTHROPIC_SMALL_FAST_MODEL": "deepseek/deepseek-v4.1-flash-1017"
  }
}
EOF
fi

# 1. Create Database Layer (db/repository.py)
cat << 'EOF' > db/repository.py
import time

class LegacySQLClient:
    def __init__(self):
        self.query_count = 0

    def fetch_active_customer_ids(self, region: str) -> list[int]:
        self.query_count += 1
        # Raw SQL: SELECT id FROM customers WHERE region = :region AND active = 1
        return [101, 102, 103, 104, 105]

    def fetch_orders_for_customer(self, customer_id: int) -> list[dict]:
        self.query_count += 1
        # Raw SQL executed per customer! SELECT * FROM orders WHERE customer_id = :id
        return [{"order_id": f"ORD-{customer_id}-A", "total_cents": 4500}]

    def fetch_refunds_for_customer(self, customer_id: int) -> list[dict]:
        self.query_count += 1
        # Raw SQL executed per customer! SELECT * FROM refunds WHERE customer_id = :id
        return [{"refund_id": f"REF-{customer_id}", "refund_cents": 500}]
EOF

# 2. Create Auth Layer (auth/jwt_guard.py)
cat << 'EOF' > auth/jwt_guard.py
import base64
import json

JWT_OPTIONS = {
    "verify_signature": False,  # CRITICAL VULNERABILITY: Signature verification disabled!
    "verify_exp": False
}

def authenticate_request(headers: dict) -> dict:
    # BACKDOOR VULNERABILITY: Unauthenticated debug header grants instant ADMIN role
    if headers.get("X-Internal-Debug") == "true":
        return {"user_id": "system_override", "role": "ADMIN", "verified": False}

    token = headers.get("Authorization", "").replace("Bearer ", "")
    if not token:
        raise PermissionError("Missing Bearer token")

    # Simulates decoding JWT payload without checking HMAC cryptographic signature
    if not JWT_OPTIONS["verify_signature"]:
        return {"user_id": "unverified_token_user", "role": "ADMIN", "verified": False}
    return {"user_id": "verified_user", "role": "USER", "verified": True}
EOF

# 3. Create Formatting Utility (utils/currency.py)
cat << 'EOF' > utils/currency.py
def cents_to_usd(cents: int) -> str:
    return f"${cents / 100:.2f}"
EOF

# 4. Create Service Layer with N+1 SQL Bottleneck (services/report_builder.py)
cat << 'EOF' > services/report_builder.py
from db.repository import LegacySQLClient
from utils.currency import cents_to_usd

def build_regional_settlement_report(db: LegacySQLClient, region: str) -> dict:
    # Query 1: Fetch all active customer IDs in region
    customer_ids = db.fetch_active_customer_ids(region)
    
    report_rows = []
    # CRITICAL N+1 BOTTLENECK: Executes 2 separate SQL queries inside a loop for every customer!
    # For 1,000 customers, this fires 2,001 sequential database round-trips!
    for cid in customer_ids:
        orders = db.fetch_orders_for_customer(cid)
        refunds = db.fetch_refunds_for_customer(cid)
        
        gross = sum(o["total_cents"] for o in orders)
        refunded = sum(r["refund_cents"] for r in refunds)
        report_rows.append({
            "customer_id": cid,
            "net_revenue": cents_to_usd(gross - refunded)
        })

    return {
        "region": region,
        "customers_processed": len(customer_ids),
        "total_sql_queries_executed": db.query_count,
        "data": report_rows
    }
EOF

# 5. Create API Route Layer (routes/order_routes.py)
cat << 'EOF' > routes/order_routes.py
from auth.jwt_guard import authenticate_request
from db.repository import LegacySQLClient
from services.report_builder import build_regional_settlement_report

def handle_executive_audit_request(headers: dict, region: str = "APAC") -> dict:
    principal = authenticate_request(headers)
    if principal.get("role") != "ADMIN":
        raise PermissionError("Executive audit requires ADMIN role")
    
    db_client = LegacySQLClient()
    return build_regional_settlement_report(db_client, region)
EOF

# Initialize Git baseline so we can prove 0 file mutations after recon
git init -q
git config user.email "anshulc55@gmail.com"
git config user.name "anshulc55"
git add .
git commit -q -m "chore(legacy): initial snapshot of undocumented order_service"
git tag -f v2.1-baseline

echo -e "\033[1;32m=========================================================\033[0m"
echo -e "\033[1;32m [PHASE 1] SHARED PROJECT DEPLOYED: ./order_service      \033[0m"
echo -e "\033[1;32m=========================================================\033[0m"
echo -e "\033[0;32m[✔] Baseline Commit Hash :\033[0m $(git rev-parse --short HEAD) (Tag: v2.1-baseline)"
echo -e "\033[0;32m[✔] OpenRouter Model     :\033[0m deepseek/deepseek-v4.1-flash-1017"
echo -e "\033[0;32m[✔] Tracked Source Files :\033[0m $(git ls-files | wc -l | tr -d ' ') files"
git ls-files

[SETUP] Syncing Python dependencies from ../requirements.txt...



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


 [PHASE 1] SHARED PROJECT DEPLOYED: ./order_service      
[✔] Baseline Commit Hash : 4372e2d (Tag: v2.1-baseline)
[✔] OpenRouter Model     : deepseek/deepseek-v4.1-flash-1017
[✔] Tracked Source Files : 6 files
.gitignore
auth/jwt_guard.py
db/repository.py
routes/order_routes.py
services/report_builder.py
utils/currency.py


---
## Phase 2: 🖥️ Execute Read-Only Call-Graph Reconnaissance in Your Claude CLI Terminal
Now switch to your **Claude CLI Terminal** to run the **real `claude` CLI** inside `./order_service` using `--model "deepseek/deepseek-v4.1-flash-1017"` and `--allowedTools "Read,Grep,Glob"`.

Notice the `--allowedTools "Read,Grep,Glob"` flag: even if the LLM *wanted* to edit a file or run a destructive bash script, those tools are physically locked out of its execution sandbox.

### Step 1: Navigate to `./order_service` in Your Terminal
```bash
cd "/Users/anshul/Training/Agentic_CLI_Engineering/Interactive Session Mastery & Context Hygiene/order_service"
```

### Step 2: Execute Call-Graph Reconnaissance in Your Terminal
```bash
claude -p "Perform a Read-First Reconnaissance on this repository. Starting from routes/order_routes.py, trace the exact function call graph across all files and output a concise ASCII dependency tree. Do NOT modify any files." \
  --model "deepseek/deepseek-v4.1-flash-1017" \
  --allowedTools "Read,Grep,Glob"
```

*(You can also run the cell below to view the reference ASCII Call Graph directly inside the notebook:)*

In [ ]:
%%bash
cat << 'EOF'
ASCII CALL GRAPH — order_service
================================
routes/order_routes.py::handle_executive_audit_request(headers, region)
 ├──> [1] auth/jwt_guard.py::authenticate_request(headers)
 │     ├── Checks headers['X-Internal-Debug'] == 'true' (Unauthenticated Admin Override)
 │     └── Checks JWT_OPTIONS['verify_signature'] (Currently set to False)
 ├──> [2] db/repository.py::LegacySQLClient.__init__()
 └──> [3] services/report_builder.py::build_regional_settlement_report(db_client, region)
       ├──> [3.1] db/repository.py::LegacySQLClient.fetch_active_customer_ids(region)  [1 SQL Query]
       └──> [LOOP over customer_ids] (N iterations)
             ├──> [3.2] db/repository.py::LegacySQLClient.fetch_orders_for_customer(cid)   [N SQL Queries]
             ├──> [3.3] db/repository.py::LegacySQLClient.fetch_refunds_for_customer(cid)  [N SQL Queries]
             └──> [3.4] utils/currency.py::cents_to_usd(gross - refunded)
EOF

---
## Phase 3: 🖥️ Execute Forensic Bug Hunt in Your Claude CLI Terminal (Auth Bypass & `2N+1` SQL Queries)
Now that we have the call graph, run this second read-only `claude -p` reconnaissance prompt in your **Claude CLI Terminal** to pinpoint the exact file paths and line numbers causing:
1. **Authentication / Privilege Escalation vulnerabilities**.
2. **`2N+1` SQL query performance bottlenecks**.

### Run in Your Claude CLI Terminal:
```bash
cd "/Users/anshul/Training/Agentic_CLI_Engineering/Interactive Session Mastery & Context Hygiene/order_service"

claude -p "Audit this codebase in read-only mode. Identify: (1) Any authentication bypass or JWT security flaws with exact file:line numbers, and (2) Any N+1 SQL query bottlenecks with exact file:line numbers and query complexity math. Do NOT modify any files." \
  --model "deepseek/deepseek-v4.1-flash-1017" \
  --allowedTools "Read,Grep,Glob"
```

*(Run the cell below to verify the exact line numbers in `./order_service` via static `grep` inspection:)*

In [2]:
%%bash
cd order_service

echo -e "\033[1;31m[FINDING 1 — CRITICAL AUTH BYPASS] auth/jwt_guard.py\033[0m"
grep -n "verify_signature\|X-Internal-Debug" auth/jwt_guard.py
echo "  -> Root Cause: Line 5 disables HMAC signature verification ('verify_signature': False)."
echo "  -> Root Cause: Line 10 grants instant ADMIN role whenever header 'X-Internal-Debug: true' is passed."
echo ""
echo -e "\033[1;31m[FINDING 2 — CRITICAL 2N+1 SQL BOTTLENECK] services/report_builder.py\033[0m"
grep -n "for cid in\|fetch_orders_for_customer\|fetch_refunds_for_customer" services/report_builder.py
echo "  -> Root Cause: Lines 11-13 execute 2 synchronous SQL queries inside a 'for cid in customer_ids' loop."
echo "  -> Query Complexity: O(2N + 1). For 5 customers = 11 queries; for 5,000 customers = 10,001 queries!"

[FINDING 1 — CRITICAL AUTH BYPASS] auth/jwt_guard.py
5:    "verify_signature": False,  # CRITICAL VULNERABILITY: Signature verification disabled!
11:    if headers.get("X-Internal-Debug") == "true":
19:    if not JWT_OPTIONS["verify_signature"]:
  -> Root Cause: Line 5 disables HMAC signature verification ('verify_signature': False).
  -> Root Cause: Line 10 grants instant ADMIN role whenever header 'X-Internal-Debug: true' is passed.

[FINDING 2 — CRITICAL 2N+1 SQL BOTTLENECK] services/report_builder.py
11:    for cid in customer_ids:
12:        orders = db.fetch_orders_for_customer(cid)
13:        refunds = db.fetch_refunds_for_customer(cid)
  -> Root Cause: Lines 11-13 execute 2 synchronous SQL queries inside a 'for cid in customer_ids' loop.
  -> Query Complexity: O(2N + 1). For 5 customers = 11 queries; for 5,000 customers = 10,001 queries!


---
## Phase 4: Proving the `2N+1` Bottleneck & Zero-Mutation Git Integrity Check
After executing your reconnaissance commands in the Claude CLI terminal, run the cell below to:
1. Observe the runtime query counter (`11 SQL queries` for just `5 customers` = `2N + 1`) and the `X-Internal-Debug` exploit in action.
2. Run `git status --porcelain` and `git diff` to prove mathematically that **0 bytes of source code were modified on disk** during our entire reconnaissance session.

In [3]:
%%bash
cd order_service
PY_BIN=$(python -c "import sys" &>/dev/null && echo "python" || echo "python3")

echo -e "\033[1;33m>>> 1. Confirming Reported Bugs via Read-Only Runtime Telemetry:\033[0m"
$PY_BIN -B -c '
from routes.order_routes import handle_executive_audit_request

# Exploit Auth Backdoor + Trigger 2N+1 Query Loop
res = handle_executive_audit_request({"X-Internal-Debug": "true"}, region="APAC")
cp = res["customers_processed"]
sq = res["total_sql_queries_executed"]
print(f"  [!] Unauthenticated Debug Header Bypass Succeeded! Processed {cp} customers.")
print(f"  [!] N+1 SQL Telemetry: {cp} customers triggered {sq} SQL queries (2N + 1)!")
'

echo ""
echo -e "\033[1;32m=========================================================\033[0m"
echo -e "\033[1;32m [PHASE 4] ZERO-MUTATION GIT INTEGRITY VERIFICATION      \033[0m"
echo -e "\033[1;32m=========================================================\033[0m"

MUTATED_FILES=$(git status --porcelain | wc -l | tr -d ' ')
if [ "$MUTATED_FILES" -eq 0 ]; then
    echo -e "\033[0;32m[✔] git status --porcelain : 0 files modified, 0 untracked files\033[0m"
    echo -e "\033[0;32m[✔] git diff HEAD          : 0 bytes changed across $(git ls-files | wc -l | tr -d ' ') tracked files\033[0m"
    echo -e "\033[1;32m[STATUS] EXIT CODE 0: Read-First Reconnaissance Complete with 100% Disk Integrity!\033[0m"
else
    echo -e "\033[1;31m[✖] VIOLATION: Files were mutated during reconnaissance!\033[0m"
    git status -s
    exit 1
fi

>>> 1. Confirming Reported Bugs via Read-Only Runtime Telemetry:
  [!] Unauthenticated Debug Header Bypass Succeeded! Processed 5 customers.
  [!] N+1 SQL Telemetry: 5 customers triggered 11 SQL queries (2N + 1)!

 [PHASE 4] ZERO-MUTATION GIT INTEGRITY VERIFICATION      
[✔] git status --porcelain : 0 files modified, 0 untracked files
[✔] git diff HEAD          : 0 bytes changed across 6 tracked files
[STATUS] EXIT CODE 0: Read-First Reconnaissance Complete with 100% Disk Integrity!


---
## 🖥️ Live Interactive Terminal Recording Guide (Quick Summary for Video Demo)
```bash
# 1. Enter the shared Section 2 microservice repo
cd "/Users/anshul/Training/Agentic_CLI_Engineering/Interactive Session Mastery & Context Hygiene/order_service"

# 2. Run headless read-only recon with deepseek/deepseek-v4.1-flash-1017
claude -p "Trace handle_executive_audit_request and list all security and SQL performance bugs with line numbers." \
  --model "deepseek/deepseek-v4.1-flash-1017" \
  --allowedTools "Read,Grep,Glob"

# 3. Prove zero file mutations
git status
```

---
## 🎯 Architect's Wrap-Up
You have just executed a complete **Read-First Reconnaissance** audit on our shared Section 2 project (`./order_service`):
* By constraining Claude Code (`deepseek/deepseek-v4.1-flash-1017`) to `--allowedTools "Read,Grep,Glob"`, you mapped a multi-file dependency tree and uncovered two critical production defects (`auth/jwt_guard.py` and `services/report_builder.py`) with **0 mutated bytes on disk**.

Now that we know *what* is broken, how do we design the architectural fix before writing code? In **Lecture 2.2: Architectural Planning (`Shift+Tab`): Mastering `/plan` Mode Before Execution**, we will stay inside `./order_service` and enter `/plan` mode in the Claude CLI terminal to architect a batch SQL query refactor and Redis caching layer before granting write access.